# Phase 5 — RAG Knowledge Base (SEC filings, Gemini embeddings)

> Run AFTER Phase 5 is built (restart the kernel first). Requires `SEC_USER_AGENT` in `.env`.

**Key facts:** only Individual Stocks file 10-K/10-Q — ETFs/cash are skipped, so fund-only clients (CLT-001/003/004) legitimately have no filings. Embeddings = `gemini-embedding-001` @ 768 dims. Free tier allows ~100 embed requests/min — ingestion sub-batches and backs off automatically.


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from dotenv import load_dotenv
load_dotenv('../.env')
assert os.getenv('GOOGLE_API_KEY'), 'Set GOOGLE_API_KEY in .env'
assert os.getenv('SEC_USER_AGENT'), 'Set SEC_USER_AGENT (name + email) in .env — EDGAR requires it'
print('env OK')

env OK


## 1. Which tickers CAN be ingested? (stocks yes, funds no)

In [2]:
from app.knowledge.ingestion import stock_tickers_in_book
stocks, skipped = stock_tickers_in_book()
print('individual stocks (ingestable):', stocks)
print('skipped (ETFs/cash — no filings):', skipped)

{"rows": 84, "clients": 10, "event": "portfolios_loaded", "level": "info", "timestamp": "2026-07-12T20:34:32.735384Z"}
individual stocks (ingestable): ['AAPL', 'ABBV', 'ADBE', 'AMZN', 'BAC', 'BRK.B', 'CRM', 'CRWD', 'CVX', 'DDOG', 'DUK', 'ENPH', 'F', 'GOOGL', 'HD', 'INTC', 'JNJ', 'JPM', 'KO', 'META', 'MSFT', 'NEE', 'NET', 'NFLX', 'NVDA', 'PATH', 'PEP', 'PG', 'PLTR', 'ROKU', 'SHOP', 'SNOW', 'T', 'TSLA', 'TWLO', 'UNH', 'VZ', 'WMT', 'XOM', 'ZM']
skipped (ETFs/cash — no filings): ['AGG', 'ARKK', 'BND', 'CASH', 'EFA', 'ESGD', 'ICLN', 'MUB', 'PBW', 'QQQ', 'SCHD', 'SCHG', 'SPY', 'VB', 'VEA', 'VOO', 'VSGX', 'VTEB', 'VTI', 'VWO', 'VXUS', 'VYM']


## 2. Ingest a couple of tickers (idempotent — re-runs skip existing)
Takes a few minutes on the free embedding tier. Skip this cell if you already ran `python -m scripts.ingest_kb`.

In [3]:
from app.knowledge.ingestion import ingest
summary = ingest(tickers=['NVDA', 'MSFT'], limit=2)
import json; print(json.dumps(summary, indent=2))

{
  "ingested": {},
  "skipped_existing": [
    "0001045810-26-000052",
    "0001045810-26-000021",
    "0001193125-26-191507",
    "0001193125-26-027207"
  ],
  "failed": {},
  "collection_size": 480
}


## 3. Collection sizes

In [4]:
from app.knowledge.vector_store import get_vector_store
vs = get_vector_store()
for c in ['sec_filings', 'news_archive', 'research_notes']:
    print(f'{c:16} {vs.count(c)} chunks')

sec_filings      480 chunks
news_archive     0 chunks
research_notes   0 chunks


## 4. Grounded retrieval with citations (Facade: Retriever)

In [5]:
from app.knowledge.retriever import Retriever
r = Retriever()
for content, cite in r.query('data center demand and revenue growth', filters={'ticker': 'NVDA'}, k=3):
    print(f"[source: {cite['form']} {cite['ticker']} {cite['filing_date']}] ({cite['section']})")
    print('   ', content[:150].replace('\n', ' '), '...\n')

HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"


{"collection": "sec_filings", "hits": 3, "filters": {"ticker": "NVDA"}, "event": "retrieval", "level": "info", "timestamp": "2026-07-12T20:35:06.629424Z"}
[source: 10-Q NVDA 2026-05-20] (Item 1A)
    Revenue was $81.6 billion, up 85% from a year ago and up 20% sequentially. Data Center revenue was $75.2 billion, up 92% from a year ago and up 21% se ...

[source: 10-Q NVDA 2026-05-20] (Item 1)
    The availability of data centers, energy, and capital to support the buildout of NVIDIA AI infrastructure by our customers and partners is crucial, an ...

[source: 10-Q NVDA 2026-05-20] (Item 1A)
    24 while ACIE addresses our growth opportunity in diverse AI purpose-built data centers and AI factories across industries and countries. Edge Computi ...



## 5. The RAG tool adds citations + the freshness disclosure line

In [6]:
from app.tools.rag_tools import search_filings
res = search_filings('data center demand', ticker='NVDA')
print('first citation      :', res['results'][0]['citation'])
print('freshness disclosure:', res['freshness_disclosure'])

HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"


{"collection": "sec_filings", "hits": 5, "filters": {"ticker": "NVDA"}, "event": "retrieval", "level": "info", "timestamp": "2026-07-12T20:35:48.231658Z"}
first citation      : [source: 10-Q NVDA 2026-05-20]
freshness disclosure: (Filing data as of 2026-05-20)


## 6. A ticker with nothing ingested answers honestly

In [7]:
print(search_filings('anything', ticker='ZZZZ')['message'])

HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"


{"collection": "sec_filings", "hits": 0, "filters": {"ticker": "ZZZZ"}, "event": "retrieval", "level": "info", "timestamp": "2026-07-12T20:38:59.733194Z"}
No relevant filings in the knowledge base for ZZZZ. Either nothing is ingested for this ticker, or it's an ETF/fund (they don't file 10-K/10-Q).


## ✅ Acceptance check

In [8]:
assert vs.count('sec_filings') > 0
assert res['results'] and 'NVDA' in res['results'][0]['citation']
assert res['freshness_disclosure'].startswith('(Filing data as of')
assert search_filings('x', ticker='ZZZZ')['results'] == []
assert not os.getenv('OPENAI_API_KEY'), 'no OpenAI key should exist anywhere'
print('Phase 5 acceptance: PASS — grounded, cited, honestly dated, Gemini-only')

HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"


{"collection": "sec_filings", "hits": 0, "filters": {"ticker": "ZZZZ"}, "event": "retrieval", "level": "info", "timestamp": "2026-07-12T20:39:12.007621Z"}
Phase 5 acceptance: PASS — grounded, cited, honestly dated, Gemini-only
